In [1]:
# 1. Instalar PySpark en el entorno de Colab
!pip install pyspark -q

import time
import pandas as pd
from pyspark.sql import SparkSession

print("Librerías instaladas y cargadas correctamente.")

Librerías instaladas y cargadas correctamente.


In [10]:
# Archivo CSV subido
nombre_archivo = "/olist_pedidos.csv"
print(f"Archivo a procesar: {nombre_archivo}")

Archivo a procesar: /olist_pedidos.csv


In [16]:
print("--- INICIANDO PRUEBA CON PANDAS (Memoria RAM) ---")
start_pandas = time.time()

# Cargar y consultar agrupando por estado del pedido
df_pandas = pd.read_csv(nombre_archivo)
resultado_pandas = df_pandas.groupby('Estado_pedido').size()

end_pandas = time.time()
tiempo_pandas = end_pandas - start_pandas
print(f"Tiempo total con Pandas: {tiempo_pandas:.4f} segundos")
print("\n--- RESULTADO DE LA CONSULTA ---")
print(resultado_pandas)

--- INICIANDO PRUEBA CON PANDAS (Memoria RAM) ---
Tiempo total con Pandas: 0.3702 segundos

--- RESULTADO DE LA CONSULTA ---
Estado_pedido
approved           2
canceled         625
created            5
delivered      96478
invoiced         314
processing       301
shipped         1107
unavailable      609
dtype: int64


In [13]:
print("--- INICIANDO PRUEBA CON PYSPARK (Procesamiento Distribuido) ---")

# Iniciar sesión Spark (El nodo Driver)
spark = SparkSession.builder.appName("OlistTest").getOrCreate()

start_spark = time.time()

# Cargar, agrupar y ejecutar la acción
df_spark = spark.read.csv(nombre_archivo, header=True, inferSchema=True)
resultado_spark = df_spark.groupBy("Estado_pedido").count()
resultado_spark.collect() # Esto detona el procesamiento en los nodos

end_spark = time.time()
tiempo_spark = end_spark - start_spark

print(f"Tiempo total con PySpark: {tiempo_spark:.4f} segundos\n")
print("--- RESULTADO DE LA CONSULTA ---")
resultado_spark.show()

--- INICIANDO PRUEBA CON PYSPARK (Procesamiento Distribuido) ---
Tiempo total con PySpark: 3.8767 segundos

--- RESULTADO DE LA CONSULTA ---
+-------------+-----+
|Estado_pedido|count|
+-------------+-----+
|      shipped| 1107|
|     canceled|  625|
|     approved|    2|
|     invoiced|  314|
|      created|    5|
|    delivered|96478|
|  unavailable|  609|
|   processing|  301|
+-------------+-----+



In [14]:
print("--- PANDAS: TOP 5 DÍAS CON MÁS PEDIDOS ---")
start_pandas_2 = time.time()

# Agrupar por fecha, contar, ordenar de mayor a menor y sacar los 5 primeros
top_fechas_pd = df_pandas.groupby('Fecha_compra').size().sort_values(ascending=False).head(5)

end_pandas_2 = time.time()
print(f"Tiempo Pandas: {end_pandas_2 - start_pandas_2:.4f} segundos\n")
print(top_fechas_pd)

--- PANDAS: TOP 5 DÍAS CON MÁS PEDIDOS ---
Tiempo Pandas: 0.0985 segundos

Fecha_compra
2/08/2018 12:06     28
2/08/2018 12:05     26
2/08/2018 12:07     22
20/11/2017 11:46    20
20/11/2017 12:47    13
dtype: int64


In [15]:
print("--- PYSPARK: TOP 5 DÍAS CON MÁS PEDIDOS ---")
start_spark_2 = time.time()

# Agrupar por fecha, contar, ordenar de forma descendente y limitar a 5
top_fechas_sp = df_spark.groupBy("Fecha_compra").count().sort("count", ascending=False).limit(5)
top_fechas_sp.collect() # Detonar el procesamiento

end_spark_2 = time.time()
print(f"Tiempo PySpark: {end_spark_2 - start_spark_2:.4f} segundos\n")
top_fechas_sp.show()

--- PYSPARK: TOP 5 DÍAS CON MÁS PEDIDOS ---
Tiempo PySpark: 2.3906 segundos

+----------------+-----+
|    Fecha_compra|count|
+----------------+-----+
| 2/08/2018 12:06|   28|
| 2/08/2018 12:05|   26|
| 2/08/2018 12:07|   22|
|20/11/2017 11:46|   20|
|20/11/2017 10:59|   13|
+----------------+-----+

